# Anomaly Detection Benchmark: Isolation Forest vs Extended IF vs Autoencoder + XAI

**Author:** Matías Fernández Lakatos  
**GitHub:** [MFLakatos](https://github.com/MFLakatos)  
**Portfolio:** [mflakatos.github.io/MFernandezLakatos.github.io](https://mflakatos.github.io/MFernandezLakatos.github.io/)

---

## Motivation

In my work at **Gradiant**, I develop unsupervised anomaly detection systems for corporate cybersecurity using real-time, large-scale data streams. This notebook is a public demonstration of the same core techniques applied to the **KDD Cup '99** network intrusion dataset — a classic benchmark for anomaly detection.

We compare three approaches:
1. **Isolation Forest (IF)** — fast, tree-based, no assumptions on data distribution
2. **Extended Isolation Forest (EIF)** — eliminates IF's known hyperplane bias
3. **Autoencoder (AE)** — deep learning approach, learns a compressed representation and flags high-reconstruction-error samples

Finally, we use **SHAP** to explain *why* specific samples are flagged — the same XAI approach used in production to make model decisions interpretable for security analysts.

---

## 1. Setup & Imports

In [ ]:
# Install dependencies (run once)
# !pip install eif shap pandas numpy scikit-learn matplotlib seaborn torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import shap
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

print('All imports OK')

## 2. Load & Explore the KDD Cup '99 Dataset

KDD Cup '99 is a benchmark for **network intrusion detection**. It contains ~5M records with 41 features (connection-level statistics) and a label column with attack types (`normal`, `dos`, `r2l`, `u2r`, `probe`).

We reframe it as a binary anomaly detection problem: **normal vs. attack**.

In [ ]:
from sklearn.datasets import fetch_kddcup99

# Load 10% sample to keep things manageable
kdd = fetch_kddcup99(subset='10percent', as_frame=True, percent10=True)
df = kdd.frame.copy()

print(f'Shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['labels'].value_counts().head(15))

In [ ]:
# ── Preprocessing ──────────────────────────────────────────────

# Binary label: 0 = normal, 1 = attack (anomaly)
df['is_anomaly'] = (df['labels'] != b'normal.').astype(int)

# Encode categorical features
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'labels']

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Feature matrix
feature_cols = [c for c in df.columns if c not in ['labels', 'is_anomaly']]
X = df[feature_cols].values.astype(np.float32)
y = df['is_anomaly'].values

# Standardise
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Features: {len(feature_cols)}')
print(f'Anomaly rate: {y.mean():.1%}')

# Train / test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3. Model 1: Isolation Forest

In [ ]:
# Isolation Forest
# contamination = expected proportion of anomalies in the data
contamination = y.mean()

if_model = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    max_features=0.8,
    n_jobs=-1,
    random_state=42
)
if_model.fit(X_train)

# Predict: IsolationForest returns -1 (anomaly) / 1 (normal)
if_preds_raw = if_model.predict(X_test)
if_preds = (if_preds_raw == -1).astype(int)  # convert to 0/1
if_scores = -if_model.decision_function(X_test)  # higher = more anomalous

print('=== Isolation Forest ===' )
print(classification_report(y_test, if_preds, target_names=['Normal', 'Anomaly']))
print(f'ROC-AUC: {roc_auc_score(y_test, if_scores):.4f}')

## 4. Model 2: Extended Isolation Forest

Standard IF has a known bias: cuts are always axis-aligned, so anomaly scores near the origin or axes are unreliable. EIF fixes this by using random hyperplane cuts.

In [ ]:
try:
    import eif
    
    eif_model = eif.iForest(
        X_train,
        ntrees=200,
        sample_size=256,
        ExtensionLevel=1  # 0 = standard IF, 1 = full EIF
    )
    
    eif_scores = eif_model.compute_paths(X_in=X_test)
    # EIF scores: higher = more anomalous (opposite to sklearn IF scores)
    threshold = np.percentile(eif_scores, (1 - contamination) * 100)
    eif_preds = (eif_scores >= threshold).astype(int)
    
    print('=== Extended Isolation Forest ===')
    print(classification_report(y_test, eif_preds, target_names=['Normal', 'Anomaly']))
    print(f'ROC-AUC: {roc_auc_score(y_test, eif_scores):.4f}')

except ImportError:
    print('eif not installed. Run: pip install eif')
    print('Using sklearn IsolationForest with ExtensionLevel workaround instead.')
    eif_preds = if_preds  # fallback
    eif_scores = if_scores

## 5. Model 3: Autoencoder (PyTorch)

The Autoencoder learns to compress and reconstruct **normal** traffic. Anomalies, being out-of-distribution, produce high reconstruction error — which becomes our anomaly score.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Linear(64, input_dim),
        )
    
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


def train_autoencoder(X_normal, epochs=30, batch_size=512, lr=1e-3):
    """Train only on normal samples — the key to unsupervised anomaly detection."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    tensor = torch.FloatTensor(X_normal).to(device)
    loader = DataLoader(TensorDataset(tensor, tensor), batch_size=batch_size, shuffle=True)
    
    model = Autoencoder(input_dim=X_normal.shape[1]).to(device)
    optimiser = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    train_losses = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, _ in loader:
            optimiser.zero_grad()
            out = model(xb)
            loss = criterion(out, xb)
            loss.backward()
            optimiser.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader)
        train_losses.append(avg_loss)
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}/{epochs}  loss: {avg_loss:.6f}')
    
    return model, train_losses, device


# Train only on normal training samples
X_train_normal = X_train[y_train == 0]
print(f'Training autoencoder on {len(X_train_normal)} normal samples...')
ae_model, ae_losses, device = train_autoencoder(X_train_normal, epochs=30)

In [ ]:
# Evaluate Autoencoder
ae_model.eval()
with torch.no_grad():
    X_test_t = torch.FloatTensor(X_test).to(device)
    recon = ae_model(X_test_t).cpu().numpy()

# Reconstruction error per sample
ae_scores = np.mean((X_test - recon) ** 2, axis=1)

# Threshold at (1 - contamination) percentile
threshold_ae = np.percentile(ae_scores, (1 - contamination) * 100)
ae_preds = (ae_scores >= threshold_ae).astype(int)

print('=== Autoencoder ===')
print(classification_report(y_test, ae_preds, target_names=['Normal', 'Anomaly']))
print(f'ROC-AUC: {roc_auc_score(y_test, ae_scores):.4f}')

## 6. Results Comparison

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = {
    'Isolation Forest':          (if_preds,  if_scores),
    'Extended Isolation Forest': (eif_preds, eif_scores),
    'Autoencoder':               (ae_preds,  ae_scores),
}

rows = []
for name, (preds, scores) in results.items():
    rows.append({
        'Model':     name,
        'Precision': precision_score(y_test, preds),
        'Recall':    recall_score(y_test, preds),
        'F1':        f1_score(y_test, preds),
        'ROC-AUC':   roc_auc_score(y_test, scores),
    })

results_df = pd.DataFrame(rows).set_index('Model')
print(results_df.round(4))

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
results_df.plot(kind='bar', ax=ax, color=['#1e3a8a','#0f766e','#7c3aed','#d97706'])
ax.set_ylim(0, 1.05)
ax.set_title('Anomaly Detection Model Comparison — KDD Cup 99', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

## 7. Explainability with SHAP

We use **SHAP (SHapley Additive exPlanations)** to understand which features drive the anomaly score for Isolation Forest — the same kind of XAI tooling applied at Gradiant to help security analysts understand model decisions.

In [ ]:
# SHAP for Isolation Forest (tree explainer is fast and exact)
print('Computing SHAP values (this may take ~1 min)...')

explainer = shap.TreeExplainer(if_model)

# Use a subsample for speed
idx = np.random.choice(len(X_test), size=500, replace=False)
shap_values = explainer.shap_values(X_test[idx])

print('Done. Plotting...')

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_test[idx],
    feature_names=feature_cols,
    max_display=15,
    show=False
)
plt.title('SHAP Feature Importance — Isolation Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150)
plt.show()

In [ ]:
# Explain a single anomaly
anomaly_indices = np.where(if_preds[idx] == 1)[0]
if len(anomaly_indices) > 0:
    i = anomaly_indices[0]
    print(f'Explaining sample {i} (predicted anomaly):')
    shap.force_plot(
        explainer.expected_value,
        shap_values[i],
        X_test[idx][i],
        feature_names=feature_cols,
        matplotlib=True
    )
    plt.savefig('shap_single_sample.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No anomalies in subsample — try increasing the subsample size.')

## 8. Anomaly Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, (preds, scores)) in zip(axes, results.items()):
    for label, color, ls in [(0, '#1e3a8a', '-'), (1, '#dc2626', '--')]:
        mask = y_test == label
        ax.hist(scores[mask], bins=60, alpha=0.5,
                color=color, linestyle=ls,
                label='Normal' if label == 0 else 'Anomaly',
                density=True)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Anomaly Score Distributions by Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('score_distributions.png', dpi=150)
plt.show()

## 9. Key Takeaways

| Model | Strengths | Weaknesses |
|---|---|---|
| **Isolation Forest** | Fast, scalable, no distributional assumptions | Axis-aligned cuts create bias in some feature spaces |
| **Extended IF** | Eliminates hyperplane bias, more robust scoring | Slightly slower, requires `eif` library |
| **Autoencoder** | Captures complex non-linear structure, learns latent representations | Requires tuning, slower to train, needs a clean normal-only training set |

**For real-time systems (like the Kafka/Spark stack at Gradiant):**  
- IF/EIF are preferred at inference time due to O(1) scoring per sample.
- Autoencoders work well as a second-stage detector or for offline batch analysis.
- SHAP explanations add ~10–100ms per sample — acceptable for async XAI pipelines, but not real-time.

---

**Author:** Matías Fernández Lakatos · [Portfolio](https://mflakatos.github.io/MFernandezLakatos.github.io/) · [LinkedIn](https://www.linkedin.com/in/mflakatos)